# RSSI distance-bucket classifier — training notebook

Two uses:
1. **Prep + sanity-check the dataset** before uploading `dataset/rssi_samples.csv` to Edge Impulse (the primary path in the project plan).
2. **Optional from-scratch path**: train a small Keras model here and INT8-quantize it to `model_int8.tflite` yourself with TFLite Micro, if you want the deeper integration story instead of the Edge Impulse export.

Requires data collected via `data_collection/log_to_csv.py` first — `dataset/rssi_samples.csv` starts out as just a header.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/rssi_samples.csv')
print(df.shape)
print(df['distance_bucket'].value_counts())
df.head()

## Feature engineering

Raw single-packet RSSI is noisy (multipath, orientation, body blocking).
Add a rolling mean/std over a short window *within each session* so the
model (or Edge Impulse's DSP block) sees a smoothed trend as well as the
instantaneous value. Window is computed per `session_id` so we never smear
across a bucket/session boundary.

In [ ]:
WINDOW = 5

df = df.sort_values(['session_id', 'timestamp_ms']).reset_index(drop=True)
grp = df.groupby('session_id')['rssi']
df['rssi_roll_mean'] = grp.transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())
df['rssi_roll_std'] = grp.transform(lambda s: s.rolling(WINDOW, min_periods=1).std().fillna(0))

feature_cols = ['rssi', 'rssi_roll_mean', 'rssi_roll_std']
df[feature_cols + ['distance_bucket']].describe()

## Train/test split by session

Split on `session_id`, not on individual rows — otherwise adjacent samples
from the same walk-and-hold session leak between train and test and the
reported accuracy will be optimistic.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = df[feature_cols].values.astype('float32')
buckets = ['near', 'mid', 'far']
y = df['distance_bucket'].map({b: i for i, b in enumerate(buckets)}).values
groups = df['session_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f'train={len(train_idx)} test={len(test_idx)}')

## Optional: train + INT8-quantize a small TFLite Micro model

Skip this section if you're exporting from Edge Impulse instead — it exists
for the manual TFLM path described as the 'alternative' in the plan.
Requires `tensorflow` (`pip install tensorflow`).

In [ ]:
import tensorflow as tf

mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-6
X_train_n = (X_train - mean) / std
X_test_n = (X_test - mean) / std

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(feature_cols),)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(len(buckets), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train_n, y_train, validation_data=(X_test_n, y_test), epochs=30, batch_size=16, verbose=2)

In [ ]:
test_loss, test_acc = model.evaluate(X_test_n, y_test, verbose=0)
print(f'Held-out test accuracy: {test_acc:.3f}')
# Record this number (and the latency/footprint measured on-device later)
# in ../results/accuracy_report.md — the plan explicitly calls out not
# estimating these numbers ahead of time.

In [ ]:
def representative_dataset():
    for row in X_train_n[:100].astype('float32'):
        yield [row.reshape(1, -1)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_int8 = converter.convert()

with open('model_int8.tflite', 'wb') as f:
    f.write(tflite_int8)
print(f'Wrote model_int8.tflite ({len(tflite_int8)} bytes)')